# Module 10 - Lab: Publish Your Data

In this lab you turn the analysed measurement into a **publication-quality data package** that a repository such as Zenodo could accept. Uploading files is not the same as publishing a dataset: a zip of files with cryptic names, no context, and no licence is technically online but practically unusable. A publication-quality package combines **data, code, docs, metadata, and a licence** that fit together and tell one consistent story.

The package format is an **RO-Crate ZIP**: a self-describing container that stores the data together with structured metadata in JSON-LD ([RO-Crate 1.3 specification](https://www.researchobject.org/ro-crate/specification/1.3/)). Lab 13 imports exactly this package when another group reuses your run.

## Learning goals

- Distinguish uploading files from publishing a dataset.
- Assemble the parts of a data package: data, metadata, docs, licence.
- Fill in the metadata record that repositories and search engines read first.
- Choose a licence deliberately and understand what BY, SA, NC, ND, and 0 mean.
- Export the package as an RO-Crate ZIP and inspect it like a reviewer.
- Prepare a data availability statement and a data citation with a persistent identifier.

<div style="border: 3px solid #b45309; background: #fff7ed; padding: 18px 20px; margin: 16px 0 24px 0; border-radius: 8px;">
<h2 style="margin-top: 0; color: #9a3412;">Prerequisite: Create the metadata first</h2>
<p>The package is built from <code>metadata.json</code>: it selects the dataset, the measurement type, and the analysis parameters that travel with the data.</p>
<p>Before you start, open <code>create_metadata_jupyter.ipynb</code> and create or review the metadata for your measurement. Ideally you have also run <code>lab_06_evaluate_measurements_jupyter.ipynb</code>, so you publish a dataset whose quality and parameters you have actually checked.</p>
</div>

## Section 1: Import Libraries

This section prepares the notebook environment. It is already prepared for you. The export and validation logic lives in `src/ro_crate_loader.py`; Lab 13 uses the same module to import the package.

In [ ]:
# Section 1: Import Libraries
from pathlib import Path
import sys

import pandas as pd

project_root = Path.cwd()
src_path = project_root / 'src'
if str(src_path) not in sys.path:
    sys.path.append(str(src_path))

from metadata_loader import load_metadata_context
from ro_crate_loader import (
    export_measurement_ro_crate_zip,
    generate_ro_crate_output_path,
)
from publication import (
    build_publication_record,
    inspect_ro_crate_deposit,
    print_publication_templates,
    review_publication_content,
    run_pre_publish_checks,
    select_license,
)

pd.set_option('display.max_columns', 30)
pd.set_option('display.max_colwidth', 120)

print('Libraries imported successfully.')

## Section 2: Review What Will Be Published

The package is built from the dataset currently selected in `metadata.json`. It will contain:

1. the **main data file** (the raw export, kept as-is),
2. any **recording metadata files** from the adjacent `meta/` folder,
3. an embedded **`metadata/metadata.json`** with only the selected use case (settings for the other measurement mode are excluded),
4. the **`ro-crate-metadata.json`** descriptor that ties everything together.

Check that this is really the dataset you want to publish before continuing. If not, change `metadata.json` with the create-metadata notebook and restart this lab.

In [ ]:
# Section 2: The dataset selected in metadata.json
metadata_context = load_metadata_context(project_root)
metadata = metadata_context['public_metadata']
selected_data_path = metadata_context['selected_data_path']

review_publication_content(metadata_context)

### Observation 1: Package Content

A reviewer should need no further explanation. Check the story the package tells.

- Is this the run you analysed in Lab 6?
- Do the file names, the metadata, and your analysis choices agree?
- Is anything missing that another group would need to reuse the data?
- Did any sensitive or personal data slip in?

- 

## Section 3: Fill In the Publication Record

Every repository asks for a **metadata record** alongside the files. It is what search engines and other researchers read first: title, description, creators, keywords, version, and licence. Title and description are derived from `metadata.json`; add the author and keywords in the **input cell** below. The second cell builds the record and shows the preview.

Identifiers connect the record: a **DOI** gives the dataset a permanent address, an **ORCID** identifies the author unambiguously. The repository assigns the DOI when you deposit; the ORCID is yours.

The **licence** states the terms of reuse. Without one, others may not legally reuse your data even when it is open. The Creative Commons elements are: **BY** credit the author, **SA** share alike, **NC** non-commercial only, **ND** no derivatives, **0** waiver of rights. The course default is **CC BY 4.0**, matching the lecture use case.

In [ ]:
# Section 3: The publication record
# ============================== YOUR INPUT ==================================
author_name = ''      # e.g. 'Group C'
author_orcid = ''     # optional, e.g. 'https://orcid.org/0000-0000-0000-0000'
keywords = [
    'phyphox',
    'RC car',
    metadata['measurement_type'],
    metadata['quantity'],
    'NFDI4ING summer school',
]
license_choice = 'CC-BY-4.0'  # 'CC-BY-4.0' | 'CC-BY-SA-4.0' | 'CC-BY-NC-4.0' | 'CC0-1.0'
# ============================================================================

In [ ]:
# Section 3: Build and preview the publication record
selected_license = select_license(license_choice)
publication_record = build_publication_record(
    metadata, author_name, author_orcid, keywords, selected_license
)

### Observation 2: Licence Choice

Justify the licence before exporting.

- Why is this licence appropriate for your data?
- What would an **NC** or **ND** element prevent another group from doing with your run?
- Who holds the rights to this data in a real project, and who must agree to the licence?

- 

## Section 4: Before You Hit Publish

A final check, mirroring the checklist from the lecture. Failed checks in the first two rows stop the export; the remaining rows are warnings you should resolve or consciously accept.

On file formats: prefer open, non-proprietary formats. The raw phyphox export is deposited **as-is** to preserve provenance; if it is a proprietary format such as `.xls`, an additional open-format copy (e.g. CSV) is recommended for preservation.

In [ ]:
# Section 4: Pre-publish checks
pre_publish_checklist = run_pre_publish_checks(
    metadata, selected_data_path, project_root, author_name, selected_license
)

### Before you pack: add your Lab 9 results (optional)

The `meta/` folder next to the selected data file is packaged automatically. If you upload your **Lab 9 results** (the documented finding, key plots, a short notes file) to JupyterHub and place them in that folder, they travel with the package. For a reuser this is the easiest possible entry: they see not only the raw numbers, but what you concluded from them and how.

Two things to decide consciously before adding anything:

- Add material only if **everyone in your group agrees** - everything in the package becomes part of the publication.
- Think twice about **videos or photos that show people** (personal data): a data publication typically stays available for **ten years or more**. Do you really want that recording to be public for that long? *"As open as possible, as closed as necessary"* - when in doubt, leave it out or plan restricted access for it instead.

## Section 5: Export the Data Package

The exporter writes the RO-Crate ZIP to `output/ro-crates/`. The file name is generated from the export date and the dataset metadata, so each dataset and version gets a distinct, sortable name. After writing, the exporter immediately re-reads the archive with the **same importer Lab 13 uses** - the package is only accepted if a reuser could actually open it.

In [ ]:
# Section 5: Export the RO-Crate ZIP
planned_path = generate_ro_crate_output_path(metadata, project_root)
print('Planned package path:', planned_path.relative_to(project_root))

ro_crate_path = export_measurement_ro_crate_zip(
    metadata,
    project_root,
    author_name=author_name or None,
    author_orcid=author_orcid or None,
    license_id=selected_license['id'],
    license_name=selected_license['name'],
    keywords=keywords,
)
print('Wrote RO-Crate package:', ro_crate_path.relative_to(project_root))
print('The archive was validated with the same importer that Lab 13 uses.')

## Section 6: Inspect the Deposit Like a Reviewer

Open your own package the way a stranger would: read only what is inside the ZIP. The summary below comes entirely from the archive - if something important is missing here, it is missing for every reuser.

In [ ]:
# Section 6: Re-import and summarize the package
inspect_ro_crate_deposit(ro_crate_path, project_root)

## Section 7: Point to the Data

In a real deposit, the repository now assigns a **DOI**. Two short texts close the loop:

- the **data availability statement** in the paper tells readers where the data lives and under what terms,
- the **data citation** lets others credit the dataset like a paper.

Choosing where to deposit is itself a decision: a **disciplinary** repository offers the best discoverability among peers, an **institutional** repository often satisfies local policy, and a **general-purpose** repository such as Zenodo accepts any discipline and issues a DOI. If unsure, check [re3data.org](https://re3data.org) and your data management plan. An **embargo** publishes the metadata record now and releases the files on a set date.

In [ ]:
# Section 7: Data availability statement and citation templates
print_publication_templates(
    publication_record, selected_license, author_name, metadata, ro_crate_path
)

### Observation 3: Deposit Decisions

Record the decisions you would make for a real deposit.

- Which repository would you choose for this dataset, and why?
- Would you publish immediately, under embargo, or restricted? What justifies the choice?
- When would you publish: with the paper, at project end, or as soon as the dataset stands on its own?
- Which related identifiers (paper, code, other versions) should the record link to?

- 